# Computer Vision --- Lecture 2 Hands-On
## Classical Image Processing


Everything in the lecture, implemented. The rule for today: **write the operator
yourself first, then check it against OpenCV.** If your version and OpenCV's
disagree, one of you has a bug -- and finding out which is the point of the exercise.

| Section | Topic |
|---|---|
| 0 | Setup and loading an image |
| 1 | Correlation and convolution from scratch |
| 2 | Separability, and how much speed it buys |
| 3 | The Gaussian: semigroup property, box vs Gaussian |
| 4 | Noise: Gaussian, salt & pepper, median, bilateral |
| 5 | Gradients: Sobel, magnitude, orientation |
| 6 | Canny, one stage at a time |
| 7 | Morphology and a complete counting pipeline |
| 8 | Features: Harris and SIFT |
| 9 | The bridge: `torch.nn.Conv2d` computes your loop |

Cells marked **TODO** are for you to fill in. Exercises are at the end.

---
## 0. Setup

Only `numpy`, `opencv-python`, `matplotlib` are needed for sections 1--8.
`scikit-image` supplies the sample images (bundled -- no download).
Section 9 needs `torch`.

In [ ]:
# !pip install opencv-python scikit-image matplotlib numpy

import numpy as np
import cv2
import matplotlib.pyplot as plt
import time

np.set_printoptions(precision=3, suppress=True, linewidth=120)
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["image.cmap"] = "gray"
print("OpenCV", cv2.__version__)

In [ ]:
def show(images, titles=None, cmaps=None, figsize=None, cols=None):
    """Display a list of images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    n = len(images)
    cols = cols or n
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols,
                             figsize=figsize or (3.4 * cols, 3.4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for i, ax in enumerate(axes):
        if i < n:
            cm = "gray" if cmaps is None else cmaps[i]
            ax.imshow(images[i], cmap=cm)
            if titles:
                ax.set_title(titles[i], fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
def load_gray(path="Figs/cat.jpg", size=512):
    """Local file if it exists, otherwise a bundled sample, otherwise synthetic."""
    import os
    if os.path.exists(path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            print(f"loaded {path}")
            return cv2.resize(img, (size, size))
    try:
        from skimage import data
        print("using skimage.data.camera()")
        return cv2.resize(data.camera(), (size, size))
    except Exception:
        pass
    print("synthesizing a test image")
    img = np.zeros((size, size), np.uint8)
    cv2.rectangle(img, (60, 60), (240, 200), 200, -1)
    cv2.circle(img, (350, 320), 90, 120, -1)
    cv2.line(img, (0, 450), (size, 400), 255, 5)
    return img

img = load_gray()
print("shape:", img.shape, "dtype:", img.dtype, "range:", img.min(), "-", img.max())
show(img, ["our working image"], figsize=(4, 4))

---
## 1. Correlation and convolution from scratch

From the lecture:

$$(h \otimes I)(m,n) = \sum_{u=-k}^{k}\sum_{v=-k}^{k} h(u,v)\, I(m+u, n+v)
\qquad\text{(correlation)}$$

$$(h * I)(m,n) = \sum_{u}\sum_{v} h(u,v)\, I(m-u, n-v)
\qquad\text{(convolution -- kernel flipped)}$$

Write the correlation with explicit loops. It will be slow. That is fine --
you only need to do it once to know what a convolutional layer is doing.

In [ ]:
def correlate2d_loops(img, kernel, border=cv2.BORDER_REFLECT_101):
    """Textbook cross-correlation. O(H*W*k^2) in Python -- slow on purpose."""
    img = img.astype(np.float64)
    kh, kw = kernel.shape
    assert kh % 2 == 1 and kw % 2 == 1, "use odd-sized kernels"
    ph, pw = kh // 2, kw // 2
    padded = cv2.copyMakeBorder(img, ph, ph, pw, pw, border)
    out = np.zeros_like(img)
    for m in range(img.shape[0]):
        for n in range(img.shape[1]):
            window = padded[m:m + kh, n:n + kw]
            out[m, n] = np.sum(window * kernel)
    return out


def convolve2d_loops(img, kernel, border=cv2.BORDER_REFLECT_101):
    """Convolution = correlation with the kernel flipped in both axes."""
    return correlate2d_loops(img, kernel[::-1, ::-1], border)

In [ ]:
# A small image so the loops finish quickly
small = cv2.resize(img, (128, 128))

box = np.ones((5, 5), np.float64) / 25.0

t0 = time.time(); mine = correlate2d_loops(small, box); t_loops = time.time() - t0
t0 = time.time(); theirs = cv2.filter2D(small.astype(np.float64), -1, box,
                                        borderType=cv2.BORDER_REFLECT_101)
t_cv2 = time.time() - t0

print(f"loops:      {t_loops*1000:8.1f} ms")
print(f"cv2:        {t_cv2*1000:8.1f} ms   ({t_loops/max(t_cv2,1e-9):.0f}x faster)")
print(f"max |diff|: {np.abs(mine - theirs).max():.3e}")

Note what we just proved: **`cv2.filter2D` computes correlation, not convolution.**
So does `torch.nn.Conv2d`, despite the name. Verify it below with an asymmetric kernel.

In [ ]:
# An asymmetric kernel makes the difference visible
asym = np.array([[0, 0, 0],
                 [1, 0, 0],
                 [0, 0, 0]], np.float64)      # shifts the image

corr = correlate2d_loops(small, asym)
conv = convolve2d_loops(small, asym)
cv2f = cv2.filter2D(small.astype(np.float64), -1, asym)

print("correlation matches cv2.filter2D :", np.allclose(corr, cv2f))
print("convolution matches cv2.filter2D :", np.allclose(conv, cv2f))
print("correlation and convolution equal:", np.allclose(corr, conv))
show([corr, conv], ["correlation (shifts one way)",
                    "convolution (shifts the other)"], figsize=(7, 3.6))

**TODO 1.** Verify that convolution is commutative but correlation is not.
Take two different asymmetric kernels `a` and `b` (3x3), and compare
`convolve2d_loops(convolve2d_loops(x, a), b)` with the same thing in the
other order. Then repeat with `correlate2d_loops`. Use a small `x` (say 32x32).

In [ ]:
# TODO 1 -- your code here
a = np.array([[0, 1, 0], [0, 0, 0], [0, 0, 2]], float)
b = np.array([[0, 0, 3], [1, 0, 0], [0, 0, 0]], float)
x = cv2.resize(img, (32, 32))

# ...

---
## 2. Separability

$$G_\sigma(x,y) = g_\sigma(x)\, g_\sigma(y)
\qquad\Rightarrow\qquad
G_\sigma * I = g_\sigma^\top * (g_\sigma * I)$$

Cost drops from $O(HWk^2)$ to $O(2HWk)$. Let's confirm both the equality and the speedup.

In [ ]:
sigma, ksize = 4.0, 25
g1 = cv2.getGaussianKernel(ksize, sigma)          # (ksize, 1)
G2 = g1 @ g1.T                                    # the full 2D kernel

print("2D kernel shape:", G2.shape)
print("rank of the 2D kernel:", np.linalg.matrix_rank(G2, tol=1e-10),
      " <- rank 1 means separable")

# singular values: one dominant value confirms rank 1
sv = np.linalg.svd(G2, compute_uv=False)
print("top 3 singular values:", sv[:3])

In [ ]:
big = cv2.resize(img, (1024, 1024)).astype(np.float64)

t0 = time.time()
full2d = cv2.filter2D(big, -1, G2)
t_2d = time.time() - t0

t0 = time.time()
sep = cv2.sepFilter2D(big, -1, g1, g1)
t_sep = time.time() - t0

print(f"full 2D  ({ksize}x{ksize}):  {t_2d*1000:7.1f} ms")
print(f"separable (2 x {ksize}):    {t_sep*1000:7.1f} ms")
print(f"speedup: {t_2d/max(t_sep,1e-9):.1f}x   (theory: {ksize**2/(2*ksize):.1f}x)")
print(f"max |difference|: {np.abs(full2d - sep).max():.3e}")

The measured speedup is usually below the theoretical $k/2$ because OpenCV already
detects separable kernels internally and because memory bandwidth, not arithmetic,
is the bottleneck at this size. The asymptotic argument still holds.

---
## 3. The Gaussian: semigroup property

$$G_{\sigma_1} * G_{\sigma_2} = G_{\sqrt{\sigma_1^2 + \sigma_2^2}}$$

Variances add. Blurring twice is one bigger blur -- which is why an image pyramid
never has to re-blur the original.

In [ ]:
s1, s2 = 3.0, 4.0
s_combined = np.sqrt(s1**2 + s2**2)          # 5.0

twice = cv2.GaussianBlur(cv2.GaussianBlur(img, (0, 0), s1), (0, 0), s2)
once  = cv2.GaussianBlur(img, (0, 0), s_combined)

diff = np.abs(twice.astype(float) - once.astype(float))
print(f"sigma1={s1}, sigma2={s2}  ->  combined sigma = {s_combined}")
print(f"max |difference|:  {diff.max():.2f} grey levels")
print(f"mean |difference|: {diff.mean():.3f} grey levels")

show([twice, once, diff], [f"blur {s1} then {s2}", f"blur once, sigma={s_combined}",
                           "|difference| (should be ~0)"],
     cmaps=["gray", "gray", "magma"])

Now box vs Gaussian. Look at the box result closely: the artefacts are
horizontal and vertical, because the kernel is a square, not a disc.

In [ ]:
box9 = cv2.blur(img, (9, 9))
gau  = cv2.GaussianBlur(img, (0, 0), 2.5)

show([img, box9, gau], ["original", "box 9x9", "Gaussian sigma=2.5"])

# zoom in on a high-frequency region to see box artefacts
sl = (slice(180, 280), slice(180, 280))
show([img[sl], box9[sl], gau[sl]], ["original (zoom)", "box (streaks)", "Gaussian (smooth)"])

---
## 4. Noise, and choosing a filter to match it

The noise model determines the right filter. Gaussian blur is the right answer
for additive Gaussian noise and the wrong answer for salt & pepper.

In [ ]:
rng = np.random.default_rng(0)

def add_gaussian_noise(img, sigma=25):
    noisy = img.astype(float) + rng.normal(0, sigma, img.shape)
    return np.clip(noisy, 0, 255).astype(np.uint8)

def add_salt_pepper(img, amount=0.05):
    out = img.copy()
    m = rng.random(img.shape)
    out[m < amount / 2] = 0
    out[m > 1 - amount / 2] = 255
    return out

noisy_g  = add_gaussian_noise(img)
noisy_sp = add_salt_pepper(img)
show([img, noisy_g, noisy_sp], ["clean", "Gaussian sigma=25", "salt & pepper 5%"])

In [ ]:
def psnr(a, b):
    mse = np.mean((a.astype(float) - b.astype(float)) ** 2)
    return 10 * np.log10(255.0**2 / mse) if mse > 0 else np.inf

results = {}
for name, noisy in [("gaussian noise", noisy_g), ("salt & pepper", noisy_sp)]:
    g = cv2.GaussianBlur(noisy, (0, 0), 2)
    m = cv2.medianBlur(noisy, 5)
    b = cv2.bilateralFilter(noisy, 9, 75, 9)
    results[name] = dict(noisy=psnr(img, noisy), gaussian=psnr(img, g),
                         median=psnr(img, m), bilateral=psnr(img, b))
    show([noisy, g, m, b], [f"{name}\nPSNR {psnr(img,noisy):.1f} dB",
                            f"Gaussian  {psnr(img,g):.1f} dB",
                            f"median    {psnr(img,m):.1f} dB",
                            f"bilateral {psnr(img,b):.1f} dB"], cols=4)

print(f"{'':16s} {'noisy':>8s} {'gaussian':>9s} {'median':>8s} {'bilateral':>10s}")
for k, v in results.items():
    print(f"{k:16s} {v['noisy']:8.1f} {v['gaussian']:9.1f} "
          f"{v['median']:8.1f} {v['bilateral']:10.1f}")

Read the table, not the images. The median wins on salt & pepper by several dB;
on Gaussian noise it is beaten by the linear filters. There is no universally
best denoiser -- the right one depends on the noise model.

---
## 5. Gradients

$$S_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1\end{bmatrix}
= \begin{bmatrix}1\\2\\1\end{bmatrix}\begin{bmatrix}-1&0&1\end{bmatrix},
\qquad
\|\nabla I\| = \sqrt{I_x^2 + I_y^2}, \quad \theta = \operatorname{atan2}(I_y, I_x)$$

In [ ]:
Sx = np.array([[-1, 0, 1],
               [-2, 0, 2],
               [-1, 0, 1]], np.float64)
Sy = Sx.T

# confirm separability by hand
smooth = np.array([[1], [2], [1]], np.float64)
deriv  = np.array([[-1, 0, 1]], np.float64)
print("Sx == outer(smooth, deriv):", np.allclose(Sx, smooth @ deriv))

Ix_mine = cv2.filter2D(img.astype(np.float64), -1, Sx)
Iy_mine = cv2.filter2D(img.astype(np.float64), -1, Sy)

Ix_cv2 = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
Iy_cv2 = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)

print("Ix matches cv2.Sobel:", np.allclose(Ix_mine, Ix_cv2))
print("Iy matches cv2.Sobel:", np.allclose(Iy_mine, Iy_cv2))

If `Iy` does *not* match, check the sign: OpenCV's y axis points **down**,
so `cv2.Sobel(..., 0, 1)` measures increasing intensity downward. Sign errors
here are the single most common gradient bug.

In [ ]:
mag = np.hypot(Ix_cv2, Iy_cv2)
ang = np.arctan2(Iy_cv2, Ix_cv2)             # radians, -pi..pi

show([Ix_cv2, Iy_cv2, mag], ["Ix (vertical edges)", "Iy (horizontal edges)",
                             "gradient magnitude"], cmaps=["gray", "gray", "magma"])

# orientation as hue, magnitude as saturation
hsv = np.zeros((*img.shape, 3), np.uint8)
hsv[..., 0] = ((np.degrees(ang) % 360) / 2).astype(np.uint8)
hsv[..., 1] = np.clip(mag / mag.max() * 900, 0, 255).astype(np.uint8)
hsv[..., 2] = 255
show(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB),
     ["hue = orientation, saturation = magnitude"], cmaps=[None], figsize=(5, 5))

**TODO 2.** Demonstrate the derivative theorem numerically:

$$\frac{\partial}{\partial x}(G_\sigma * I) = \left(\frac{\partial G_\sigma}{\partial x}\right) * I$$

Build the derivative-of-Gaussian kernel $\partial_x G_\sigma(x,y) = -\frac{x}{\sigma^2} G_\sigma(x,y)$
explicitly, convolve the image with it once, and compare against
blurring first and then applying `cv2.Sobel`. They will not be bit-identical
(Sobel is only an approximation to $\partial_x$) -- report the correlation coefficient
between the two results and explain the residual.

In [ ]:
# TODO 2 -- your code here
sigma = 2.0
half = int(3 * sigma)
xs = np.arange(-half, half + 1)
X, Y = np.meshgrid(xs, xs)

# G = ...
# dGx = ...
# ...

---
## 6. Canny, stage by stage

1. Smooth with $G_\sigma$
2. Gradient magnitude and orientation
3. Non-maximum suppression along $\theta$
4. Hysteresis thresholding

Implement 3 and 4 yourself, then compare with `cv2.Canny`.

In [ ]:
def non_max_suppression(mag, ang_deg):
    """Keep a pixel only if it is a maximum along the gradient direction."""
    H, W = mag.shape
    out = np.zeros_like(mag)
    a = ang_deg % 180                       # direction, not sign
    for i in range(1, H - 1):
        for j in range(1, W - 1):
            t = a[i, j]
            if t < 22.5 or t >= 157.5:      # gradient horizontal -> compare left/right
                p, q = mag[i, j - 1], mag[i, j + 1]
            elif t < 67.5:                   # 45 degrees
                p, q = mag[i + 1, j - 1], mag[i - 1, j + 1]
            elif t < 112.5:                  # gradient vertical -> compare up/down
                p, q = mag[i - 1, j], mag[i + 1, j]
            else:                            # 135 degrees
                p, q = mag[i - 1, j - 1], mag[i + 1, j + 1]
            if mag[i, j] >= p and mag[i, j] >= q:
                out[i, j] = mag[i, j]
    return out


def hysteresis(nms, low, high):
    """Keep strong pixels, plus weak pixels connected to a strong one."""
    strong = (nms >= high).astype(np.uint8)
    weak   = ((nms >= low) & (nms < high)).astype(np.uint8)
    # connected components over strong|weak; keep a component if it holds a strong pixel
    n, labels = cv2.connectedComponents(((strong | weak) * 255).astype(np.uint8))
    keep = np.zeros(n, bool)
    keep[labels[strong == 1]] = True
    keep[0] = False                                    # background
    return (keep[labels] * 255).astype(np.uint8)

In [ ]:
sigma = 1.4
sm = cv2.GaussianBlur(img, (0, 0), sigma)
gx = cv2.Sobel(sm, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(sm, cv2.CV_64F, 0, 1, ksize=3)
magn = np.hypot(gx, gy)
angd = np.degrees(np.arctan2(gy, gx))

nms  = non_max_suppression(magn, angd)
mine = hysteresis(nms, low=60, high=140)

show([sm, magn, nms, mine], ["1. smoothed", "2. magnitude",
                             "3. after NMS", "4. after hysteresis"],
     cmaps=["gray", "magma", "magma", "gray"], cols=4)

Now compare with OpenCV. Two details matter, and they are worth more than
the comparison itself:

* `cv2.Canny` does **no pre-smoothing** -- it applies Sobel directly. Feed it
  the raw image and you are comparing against a different algorithm.
* By default it uses the **L1** magnitude $|I_x| + |I_y|$, not $\sqrt{I_x^2+I_y^2}$.
  Pass `L2gradient=True` to match ours.

In [ ]:
variants = {
    "cv2.Canny(img, L1)":  cv2.Canny(img, 60, 140),
    "cv2.Canny(img, L2)":  cv2.Canny(img, 60, 140, L2gradient=True),
    "cv2.Canny(smoothed, L1)": cv2.Canny(sm, 60, 140),
    "cv2.Canny(smoothed, L2)": cv2.Canny(sm, 60, 140, L2gradient=True),
}
print(f"ours: {(mine>0).sum()} edge pixels\n")
print(f"{'variant':26s} {'edge px':>8s} {'agreement':>10s}")
for name, e in variants.items():
    print(f"{name:26s} {int((e>0).sum()):8d} "
          f"{np.mean((mine>0)==(e>0))*100:9.2f}%")

best = variants["cv2.Canny(smoothed, L2)"]
show([mine, best, (((mine > 0) != (best > 0)) * 255).astype(np.uint8)],
     ["ours", "cv2, smoothed + L2", "disagreement"])

Matching the two conventions takes agreement from ~89% to ~99%. The residual
is our rounded NMS: OpenCV interpolates between the two neighbours instead of
snapping $\theta$ to one of four directions, so it places edges slightly differently.

In [ ]:
# how sensitive is the result to the thresholds?
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, (lo, hi) in zip(axes, [(20, 40), (60, 140), (120, 240), (200, 350)]):
    ax.imshow(cv2.Canny(img, lo, hi))
    ax.set_title(f"({lo}, {hi})", fontsize=10)
    ax.axis("off")
plt.tight_layout(); plt.show()

---
## 7. Morphology

$$A \ominus B = \{z : B_z \subseteq A\} \qquad A \oplus B = \{z : \hat{B}_z \cap A \neq \emptyset\}$$

$$A \circ B = (A \ominus B) \oplus B \qquad A \bullet B = (A \oplus B) \ominus B$$

In [ ]:
# a binary test image with a thin bridge, a hole, and specks
bw = np.zeros((240, 320), np.uint8)
cv2.rectangle(bw, (30, 40), (110, 200), 255, -1)
cv2.circle(bw, (200, 90), 45, 255, -1)
cv2.line(bw, (150, 180), (300, 180), 255, 3)
cv2.circle(bw, (200, 90), 12, 0, -1)
m = rng.random(bw.shape)
bw[m < 0.02] = 255
bw[m > 0.985] = 0

k = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
show([bw, cv2.erode(bw, k), cv2.dilate(bw, k)],
     ["input", "erosion (specks and bridge gone)", "dilation (hole filled)"])

opened = cv2.morphologyEx(bw, cv2.MORPH_OPEN, k)
closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, k)
show([bw, opened, closed], ["input", "opening", "then closing"])

In [ ]:
# duality: eroding the object == dilating the background
lhs = cv2.erode(bw, k)
rhs = 255 - cv2.dilate(255 - bw, k)
print("(A erode B)^c == A^c dilate B_hat :", np.array_equal(lhs, rhs))

# idempotence: opening twice changes nothing
print("opening is idempotent:",
      np.array_equal(opened, cv2.morphologyEx(opened, cv2.MORPH_OPEN, k)))

# structuring element shape matters
for shape, name in [(cv2.MORPH_RECT, "rect"), (cv2.MORPH_ELLIPSE, "ellipse"),
                    (cv2.MORPH_CROSS, "cross")]:
    se = cv2.getStructuringElement(shape, (9, 9))
    print(f"\n{name}:\n{se}")

### A complete classical pipeline: count the coins

No training data, no GPU. For a controlled scene this is still the right tool.

In [ ]:
from skimage import data
coins = data.coins()

blur = cv2.GaussianBlur(coins, (0, 0), 2)
_, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
clean = cv2.morphologyEx(th, cv2.MORPH_OPEN, k5, iterations=1)
clean = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, k5, iterations=1)

n, labels, stats, centroids = cv2.connectedComponentsWithStats(clean)
areas = stats[1:, cv2.CC_STAT_AREA]
big = areas > 300                                   # drop leftover specks

print(f"components found: {n-1}, after area filter: {big.sum()}  (truth: 24)")
show([coins, th, clean, (labels * 37 % 255).astype(np.uint8)],
     ["input", "Otsu threshold", "open + close", f"{big.sum()} components"],
     cmaps=["gray", "gray", "gray", "nipy_spectral"], cols=4)

It gets close but not exactly 24 -- **touching coins merge into one component**,
and more aggressive closing merges more of them (try `iterations=3` and watch the
count collapse). That failure is the honest lesson: classical pipelines break at
object contact, which is precisely why instance segmentation exists. The classical
patch is the watershed transform (`cv2.watershed`) seeded by a distance transform;
the modern one is Mask R-CNN.

---
## 8. Features: Harris and SIFT

$$M = \sum w(x,y)\begin{bmatrix} I_x^2 & I_xI_y \\ I_xI_y & I_y^2\end{bmatrix},
\qquad R = \det M - k(\operatorname{tr} M)^2$$

In [ ]:
def harris_response(gray, sigma=1.5, k=0.04):
    """Harris R computed from the structure tensor, step by step."""
    g = gray.astype(np.float64)
    Ix = cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=3)

    # products of derivatives, then a Gaussian window w
    Ixx = cv2.GaussianBlur(Ix * Ix, (0, 0), sigma)
    Iyy = cv2.GaussianBlur(Iy * Iy, (0, 0), sigma)
    Ixy = cv2.GaussianBlur(Ix * Iy, (0, 0), sigma)

    det   = Ixx * Iyy - Ixy ** 2
    trace = Ixx + Iyy
    return det - k * trace ** 2


def peaks(R, rel_thresh=0.05, nms_size=15):
    """Threshold, then keep local maxima (dilate == max-filter, section 7)."""
    mask = R > rel_thresh * R.max()
    dil = cv2.dilate(R, np.ones((nms_size, nms_size)))
    return np.argwhere(mask & (R >= dil))          # (row, col) pairs

In [ ]:
shapes = np.zeros((256, 256), np.uint8)
cv2.rectangle(shapes, (40, 40), (140, 120), 255, -1)
cv2.rectangle(shapes, (150, 140), (220, 215), 255, -1)
cv2.circle(shapes, (70, 190), 35, 255, -1)

R = harris_response(shapes)
pts = peaks(R)

vis = cv2.cvtColor(shapes, cv2.COLOR_GRAY2RGB)
for y, x in pts:
    cv2.circle(vis, (int(x), int(y)), 4, (255, 0, 0), 1)

print(f"{len(pts)} detections")
print("two rectangles contribute 8 true corners; the rest sit on the circle,")
print("where the boundary curves fast enough to look like a corner locally.")
show([shapes, R, vis], ["input", "Harris response R", "detected corners"],
     cmaps=["gray", "coolwarm", None])

### Is it really invariant? Measure, do not assume

Detect on the original, transform the detections back into the original frame,
and count how many land within 3 px of an original detection. That number --
**repeatability** -- is how the feature-detection literature actually evaluates
a detector.

In [ ]:
base = peaks(harris_response(img))

def repeatability(pts_mapped, pts_base, tol=3):
    if len(pts_mapped) == 0:
        return 0
    d = np.linalg.norm(pts_base[:, None, :] - pts_mapped[None, :, :], axis=2)
    return int((d.min(axis=1) <= tol).sum())

print(f"baseline: {len(base)} corners\n")
print(f"{'transform':22s} {'detected':>9s} {'repeatable':>11s}")

# brightness change
bright = np.clip(img * 0.6 + 40, 0, 255).astype(np.uint8)
p = peaks(harris_response(bright))
print(f"{'0.6*I + 40':22s} {len(p):9d} {repeatability(p.astype(float), base):8d}/{len(base)}")

# rotation
for a in [15, 30, 45]:
    M = cv2.getRotationMatrix2D((256, 256), a, 1.0)
    p = peaks(harris_response(cv2.warpAffine(img, M, (512, 512))))
    Minv = cv2.invertAffineTransform(M)
    xy = np.stack([p[:, 1], p[:, 0]], 1).astype(float)
    back = (Minv[:, :2] @ xy.T).T + Minv[:, 2]
    back = np.stack([back[:, 1], back[:, 0]], 1)
    print(f"{'rotate ' + str(a) + ' deg':22s} {len(p):9d} "
          f"{repeatability(back, base):8d}/{len(base)}")

# scale
for s in [1.5, 2.0, 3.0]:
    p = peaks(harris_response(cv2.resize(img, None, fx=s, fy=s,
                                         interpolation=cv2.INTER_CUBIC)))
    print(f"{'scale ' + str(s) + 'x':22s} {len(p):9d} "
          f"{repeatability(p / s, base):8d}/{len(base)}")

Read the **detected** column, not just repeatability. Brightness and rotation
leave the detection count essentially unchanged -- the detector finds the same
structures. Scaling does not: the count grows roughly with area, because the
fixed window keeps finding new fine-scale structure. The two feature *sets* are
no longer in correspondence, which is what breaks matching.

The fix is to give the detector a notion of size. A blob of radius $r$ produces
the strongest **scale-normalized** LoG response, $\sigma^2\nabla^2 L$, at
$\sigma = r/\sqrt{2}$ -- so searching over $\sigma$ recovers the object's own scale.
Confirm it:

In [ ]:
print(f"{'blob radius':>12s} {'argmax sigma':>13s} {'r/sqrt(2)':>11s}")
for r in [8, 16, 32]:
    blob = np.zeros((256, 256), np.float64)
    cv2.circle(blob, (128, 128), r, 1.0, -1)
    responses = []
    sigmas = np.arange(2, 40, 0.5)
    for s in sigmas:
        L = cv2.GaussianBlur(blob, (0, 0), s)
        responses.append(-(s**2) * cv2.Laplacian(L, cv2.CV_64F)[128, 128])
    peak = sigmas[int(np.argmax(responses))]
    print(f"{r:12d} {peak:13.1f} {r/np.sqrt(2):11.1f}")
    plt.plot(sigmas, responses, label=f"r = {r}")

plt.axhline(0, color="0.8", lw=0.8)
plt.xlabel("sigma"); plt.ylabel(r"scale-normalized LoG at the blob centre")
plt.title("Each blob peaks at its own characteristic scale")
plt.legend(); plt.show()

That is the whole idea behind SIFT's first stage, and the reason its keypoints
carry a scale as well as a position.

### SIFT: detection, description, matching

The 128-D descriptor and the ratio test. Note SIFT has been in the main OpenCV
distribution since the patent expired in 2020 -- no `contrib` build needed.

In [ ]:
from skimage import data
astro = cv2.cvtColor(data.astronaut(), cv2.COLOR_RGB2GRAY)

sift = cv2.SIFT_create(nfeatures=400)
kp, des = sift.detectAndCompute(astro, None)
print(f"{len(kp)} keypoints, descriptor shape {des.shape}, dtype {des.dtype}")
print(f"descriptor norm (should be ~1 x 512 after OpenCV's scaling): "
      f"{np.linalg.norm(des[0]):.1f}")

vis = cv2.drawKeypoints(astro, kp, None,
                        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
show(vis, ["SIFT keypoints: circle = scale, radius = orientation"],
     cmaps=[None], figsize=(6, 6))

In [ ]:
# rotate 35 degrees and scale to 0.7 -- then try to match
M = cv2.getRotationMatrix2D((256, 256), 35, 0.7)
warped = cv2.warpAffine(astro, M, (512, 512))
kp2, des2 = sift.detectAndCompute(warped, None)

bf = cv2.BFMatcher()
raw = bf.knnMatch(des, des2, k=2)

for ratio in [0.5, 0.75, 0.9, 1.0]:
    good = [m for m, n in raw if m.distance < ratio * n.distance]
    print(f"ratio < {ratio:4.2f}:  {len(good):4d} matches kept "
          f"of {len(raw)} candidates")

good = sorted([m for m, n in raw if m.distance < 0.75 * n.distance],
              key=lambda m: m.distance)[:40]
out = cv2.drawMatches(astro, kp, warped, kp2, good, None,
                      flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
show(out, [f"{len(good)} matches after the ratio test"], cmaps=[None], figsize=(12, 6))

---
## 9. The bridge: `torch.nn.Conv2d` is your loop

The last cell of the lecture. A convolutional layer with fixed weights computes
exactly the correlation you wrote in section 1. The only difference in a real
network is that the weights are learned.

*(Needs `torch`. Skip if it is not installed.)*

In [ ]:
import torch
import torch.nn.functional as F

x = torch.from_numpy(small.astype(np.float32))[None, None]     # (N, C, H, W)
w = torch.from_numpy(Sx.astype(np.float32))[None, None]        # (out, in, kh, kw)

torch_out = F.conv2d(x, w, padding=1)[0, 0].numpy()
cv2_out   = cv2.filter2D(small.astype(np.float64), -1, Sx,
                         borderType=cv2.BORDER_CONSTANT)

interior = (slice(1, -1), slice(1, -1))       # ignore the border rows/cols
print("torch.conv2d == cv2.filter2D (interior):",
      np.allclose(torch_out[interior], cv2_out[interior], atol=1e-3))
print("=> torch.nn.Conv2d computes CORRELATION, not convolution.")

show([small, torch_out, cv2_out], ["input", "F.conv2d with Sobel-x", "cv2.filter2D"])

In [ ]:
# What a conv layer looks like before it has learned anything
layer = torch.nn.Conv2d(in_channels=1, out_channels=8, kernel_size=5, padding=2)
print("weight shape:", tuple(layer.weight.shape),
      " = (out_channels, in_channels, kh, kw)")
print("parameters:", sum(p.numel() for p in layer.parameters()))

with torch.no_grad():
    feats = layer(x)[0].numpy()
show(list(feats), [f"random filter {i}" for i in range(8)], cols=4)

Random filters produce noise. After training on a classification task, the
first-layer filters of a CNN reliably converge to oriented edge and blob
detectors -- rediscovering, from data alone, the operators we spent this
lecture designing by hand. That is where Week 4 picks up.

---
## Exercises

**1. Unsharp masking.** Implement $I_{\text{sharp}} = I + \alpha(I - G_\sigma * I)$.
Sweep $\alpha \in \{0.5, 1, 2, 4\}$ and $\sigma \in \{1, 3, 6\}$ on a 3x4 grid.
At what point does it start amplifying noise instead of detail? Then show that
this is equivalent to convolving with a single kernel, and write that kernel down.

**2. Fix the aliasing bug.** Downsample the image by 8 in two ways: `img[::8, ::8]`
and blur-then-subsample. Do it on a radial chirp
(`r = np.hypot(x-256, y-256); chirp = 127*(1+np.sin(r**2/320))`) where the moire
is unmistakable. Then measure the difference quantitatively: compare the 2D FFT
magnitude of both results and explain which frequencies got folded where.

**3. Corner detection under transformation.** Take an image, apply (a) a rotation,
(b) a 2x scaling, (c) a brightness change `I -> 0.6*I + 40`. For each, run Harris
and report how many of the original corners are re-detected within 3 pixels of
their transformed position. Which transformation breaks it, and does that match
what the lecture claimed?

**4. Your own SIFT-lite.** Build a 128-D descriptor: take a 16x16 patch around a
Harris corner, split into 4x4 cells, compute an 8-bin orientation histogram per
cell weighted by gradient magnitude, concatenate, and L2-normalize. Do *not*
implement scale or orientation assignment. Match your descriptors between an
image and a rotated copy. Quantify how much worse you do than real SIFT -- the gap
is exactly the value of the orientation assignment step.

**5. Compare edge detectors on real data.** Run Sobel-with-threshold, LoG
zero-crossings, and Canny on five images of your choice. There is no ground truth,
so judge them on: (a) do the edges form closed contours? (b) how many parameters
did you have to tune per image? (c) does one setting work across all five?
Write a paragraph on why (c) is the hard part -- and what a learned boundary
detector does differently.